In [8]:
import pandas as pd

# 读取原始数据集
df = pd.read_csv('summerOly_athletes_filtered.csv')

# 标准化奖牌字段
medal_mapping = {
    'gold': 'Gold',
    'silver': 'Silver',
    'bronze': 'Bronze',
    'no medal': 'No medal'
}
df['Medal'] = df['Medal'].str.lower().map(medal_mapping).fillna('No medal')

# 创建评分映射字典
score_map = {
    'Gold': 3,
    'Silver': 2,
    'Bronze': 1,
    'No medal': 0
}

# 计算每条记录的奖牌得分
df['Medal Score'] = df['Medal'].map(score_map)

# 计算每个运动员的总分
athlete_scores = df.groupby('Name')['Medal Score'].sum().reset_index()
athlete_scores.rename(columns={'Medal Score': 'Grading by medals'}, inplace=True)

# 合并回原始数据集
df = pd.merge(df, athlete_scores, on='Name', how='left')

# 删除临时计算的得分列
df.drop('Medal Score', axis=1, inplace=True)

# 保存到新文件
df.to_csv('athletes_with_grading.csv', index=False)

print("文件已成功保存，新增的评分列位于最右侧。")

文件已成功保存，新增的评分列位于最右侧。


In [9]:
import pandas as pd

# 读取含奖牌评分的数据
df = pd.read_csv('athletes_with_grading.csv')

# 生成运动员参赛时序
year_sequence = (
    df.groupby('Name')['Year']
    .apply(lambda x: sorted(x.unique()))  # 获取唯一年份并排序
    .reset_index(name='YearList')
)

# 展开年份序列并计算经验值
year_sequence = year_sequence.explode('YearList').reset_index(drop=True)
year_sequence['Grading by experience'] = year_sequence.groupby('Name').cumcount()

# 合并回原始数据
df = pd.merge(
    df,
    year_sequence.rename(columns={'YearList': 'Year'}),
    on=['Name', 'Year'],
    how='left'
)

# 处理从未参赛的特殊情况
df['Grading by experience'] = df['Grading by experience'].fillna(0).astype(int)

# 保存到新文件
df.to_csv('athletes_with_experience_grading.csv', index=False)

print("参赛经验评分已成功添加，文件已保存。")

参赛经验评分已成功添加，文件已保存。


In [12]:
import pandas as pd

# 读取最新数据
df = pd.read_csv('athletes_with_experience_grading.csv')

# ==== 关键修正步骤 ====
# 先对运动员-届次维度去重
unique_scores = df.drop_duplicates(['NOC', 'Year', 'Name'])

# 计算国家每届总评分
country_grading = (
    unique_scores.groupby(['NOC', 'Year'])['Grading Final']
    .sum()
    .reset_index()
    .rename(columns={'Grading Final': 'Grading by Con'})
)
# ======================

# 合并回原始数据
df = pd.merge(
    df,
    country_grading,
    on=['NOC', 'Year'],
    how='left'
)

# 验证输出示例
sample_data = df[(df['NOC'] == 'CHN') & (df['Year'] == 2012)].head(2)
print("验证示例：\n", sample_data[['Name', 'Year', 'Grading Final', 'Grading by Con']])

# 保存文件
df.to_csv('olympic_grading_final_corrected.csv', index=False)

验证示例：
            Name  Year  Grading Final  Grading by Con
448  Cheng Ming  2012              2             484
449  Cheng Ming  2012              2             484


In [13]:
import pandas as pd

# 读取包含国家评分的最新文件
input_path = 'olympic_grading_final_corrected.csv'
output_path = 'olympic_grading_normalized.csv'
df = pd.read_csv(input_path)

def safe_normalize(group):
    """带防护机制的归一化函数"""
    min_val = group.min()
    max_val = group.max()
    
    # 处理全零或常数值情况
    if max_val == min_val:
        return pd.Series([0.5]*len(group), index=group.index)
    
    return (group - min_val) / (max_val - min_val)

# 分年度进行归一化
df['Grading by Con (Normalized)'] = (
    df.groupby('Year')['Grading by Con']
    .transform(safe_normalize)
    .round(4)  # 保留4位小数
)

# 验证极端情况
print("归一化范围验证：")
print(df.groupby('Year')['Grading by Con (Normalized)'].agg(['min','max']))

# 保存结果
df.to_csv(output_path, index=False)
print(f"\n文件已保存至 {output_path}，新增归一化评分列。")

归一化范围验证：
      min  max
Year          
1896  0.0  1.0
1900  0.0  1.0
1904  0.0  1.0
1906  0.0  1.0
1908  0.0  1.0
1912  0.0  1.0
1920  0.0  1.0
1924  0.0  1.0
1928  0.0  1.0
1932  0.0  1.0
1936  0.0  1.0
1948  0.0  1.0
1952  0.0  1.0
1956  0.0  1.0
1960  0.0  1.0
1964  0.0  1.0
1968  0.0  1.0
1972  0.0  1.0
1976  0.0  1.0
1980  0.0  1.0
1984  0.0  1.0
1988  0.0  1.0
1992  0.0  1.0
1996  0.0  1.0
2000  0.0  1.0
2004  0.0  1.0
2008  0.0  1.0
2012  0.0  1.0
2016  0.0  1.0
2020  0.0  1.0
2024  0.0  1.0

文件已保存至 olympic_grading_normalized.csv，新增归一化评分列。


In [14]:
import pandas as pd

# 读取数据
input_path = 'olympic_grading_normalized.csv'
output_path = 'olympic_with_athlete_counts.csv'
df = pd.read_csv(input_path)

# 计算每个运动员在每届参赛的项目数
athlete_events = df.groupby(['NOC', 'Year', 'Name']).size().reset_index(name='EventCount')

# 计算每届每个国家的总参赛项目数
country_counts = (
    athlete_events.groupby(['NOC', 'Year'])['EventCount']
    .sum()
    .reset_index()
    .rename(columns={'EventCount': 'Athlete Count by Events'})
)

# 合并回原始数据
df = pd.merge(df, country_counts, on=['NOC', 'Year'], how='left')

# 验证示例
print("\n验证示例（假设某运动员参加多项目）：")
sample_athlete = df[df['Name'] == 'Michael Phelps'].head(1)
print(sample_athlete[['NOC', 'Year', 'Sport', 'Event', 'Athlete Count by Events']])

# 保存结果
df.to_csv(output_path, index=False)
print(f"\n新增参赛人数列已保存至 {output_path}")


验证示例（假设某运动员参加多项目）：
Empty DataFrame
Columns: [NOC, Year, Sport, Event, Athlete Count by Events]
Index: []

新增参赛人数列已保存至 olympic_with_athlete_counts.csv


In [15]:
import pandas as pd

# 读取最新数据文件
input_path = 'olympic_with_athlete_counts.csv'
output_path = 'olympic_with_athlete_counts_final.csv'
df = pd.read_csv(input_path)

# 计算每届每个国家的实际参赛人数（去重计数）
athlete_counts = (
    df.groupby(['NOC', 'Year'])['Name']
    .nunique()  # 关键函数：统计不重复的姓名数量
    .reset_index()
    .rename(columns={'Name': 'Athlete Count'})
)

# 合并到原始数据
df = pd.merge(df, athlete_counts, on=['NOC', 'Year'], how='left')

# 验证示例：同一运动员多项目只计1次
sample_data = df[(df['NOC'] == 'USA') & (df['Year'] == 2016)].head(3)
print("验证示例：\n", sample_data[['Name', 'Year', 'Athlete Count']])

# 保存结果
df.to_csv(output_path, index=False)
print(f"\n新增参赛人数列已保存至 {output_path}")

验证示例：
                 Name  Year  Athlete Count
431  Mackenzie Brown  2016            271
432    Brady Ellison  2016            271
433    Brady Ellison  2016            271

新增参赛人数列已保存至 olympic_with_athlete_counts_final.csv


In [16]:
import pandas as pd

# 读取最新数据文件
input_path = 'olympic_with_athlete_counts_final.csv'
output_path = 'olympic_with_gold_counts.csv'
df = pd.read_csv(input_path)

# 统计金牌数量
gold_counts = (
    df[df['Medal'] == 'Gold']          # 筛选金牌记录
    .groupby(['NOC', 'Year'])          # 按国家-年份分组
    .size()                            # 计数
    .reset_index(name='Gold Medals')   # 重命名结果列
)

# 合并到原始数据（保留所有记录）
df = pd.merge(df, gold_counts, on=['NOC', 'Year'], how='left')

# 处理未获奖情况（填充0）
df['Gold Medals'] = df['Gold Medals'].fillna(0).astype(int)

# 验证示例（假设中国在2020年获得38金）
sample_china = df[(df['NOC'] == 'CHN') & (df['Year'] == 2020)].head(1)
print("验证示例：\n", sample_china[['NOC', 'Year', 'Gold Medals']])

# 保存结果
df.to_csv(output_path, index=False)
print(f"\n新增金牌数列已保存至 {output_path}")

验证示例：
      NOC  Year  Gold Medals
186  CHN  2020           41

新增金牌数列已保存至 olympic_with_gold_counts.csv


In [17]:
import pandas as pd

# 读取最新数据文件
input_path = 'olympic_with_gold_counts.csv'
output_path = 'olympic_with_total_medals.csv'
df = pd.read_csv(input_path)

# 统计总奖牌数（包含金/银/铜）
total_medals = (
    df[df['Medal'].isin(['Gold', 'Silver', 'Bronze'])]  # 筛选有奖牌记录
    .groupby(['NOC', 'Year'])                           # 按国家-年份分组
    .size()                                             # 计数
    .reset_index(name='Total Medals')                  # 重命名结果列
)

# 合并到原始数据
df = pd.merge(df, total_medals, on=['NOC', 'Year'], how='left')

# 处理无奖牌情况（填充0）
df['Total Medals'] = df['Total Medals'].fillna(0).astype(int)

# 验证示例（假设美国2016年获得46金+37银+38铜=121枚）
sample_usa = df[(df['NOC'] == 'USA') & (df['Year'] == 2016)].head(1)
print("验证示例：\n", sample_usa[['NOC', 'Year', 'Gold Medals', 'Total Medals']])

# 保存结果
df.to_csv(output_path, index=False)
print(f"\n新增总奖牌数列已保存至 {output_path}")

验证示例：
      NOC  Year  Gold Medals  Total Medals
431  USA  2016           47           124

新增总奖牌数列已保存至 olympic_with_total_medals.csv
